In [1]:
# --- imports ---
import pandas as pd
import sqlite3

print("All imports OK")

All imports OK


In [2]:
# --- config ---
MODE = "subway"
DATASET = "../../data/ttc-subway-delay-data-2025.csv"
LOOKUP = "../../data/ttc-subway-delay-data-lookup.csv"
DB = "../../sql/TTC_delay.db"

In [3]:
df = pd.read_csv(DATASET)
df.head()

,_id,Date,Time,Day,Station,Code,Min Delay,Min Gap,Bound,Line,Vehicle
0,1,2025-01-01,02:10,Wednesday,BATHURST STATION,MUSAN,5,9,E,BD,5227
1,2,2025-01-01,02:30,Wednesday,DUNDAS STATION,MUIRS,0,0,NaN,YU,0
2,3,2025-01-01,02:32,Wednesday,BROADVIEW STATION,PUMST,0,0,E,BD,0
3,4,2025-01-01,02:58,Wednesday,KEELE STATION,EUSC,0,0,W,BD,5293
4,5,2025-01-01,02:58,Wednesday,COXWELL STATION,SUAE,0,0,NaN,BD,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45475 entries, 0 to 45474
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   _id        45475 non-null  int64 
 1   Date       45475 non-null  object
 2   Time       45475 non-null  object
 3   Day        45475 non-null  object
 4   Station    45475 non-null  object
 5   Code       45475 non-null  object
 6   Min Delay  45475 non-null  int64 
 7   Min Gap    45475 non-null  int64 
 8   Bound      29615 non-null  object
 9   Line       45298 non-null  object
 10  Vehicle    45475 non-null  int64 
dtypes: int64(4), object(7)
memory usage: 3.8+ MB


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isna().sum()

_id              0
Date             0
Time             0
Day              0
Station          0
Code             0
Min Delay        0
Min Gap          0
Bound        15860
Line           177
Vehicle          0
dtype: int64

In [7]:
# catching stray values
df["Bound"].value_counts(dropna=False)

Bound
NaN    15860
S       7989
E       7516
W       7108
N       6940
B         62
Name: count, dtype: int64

In [8]:
valid_directions = ['N', 'S', 'E', 'W']

#  group anything that is NOT a valid direction (including NaNs) and set it to "Unknown"
df.loc[~df['Bound'].isin(valid_directions), 'Bound'] = 'Unknown'

# recheck
df["Bound"].value_counts(dropna=False)

Bound
Unknown    15922
S           7989
E           7516
W           7108
N           6940
Name: count, dtype: int64

In [9]:
# standardize column names
df = df.rename(columns={"_id": "source_id",
                        "Min Delay": "min_delay", "Min Gap": "min_gap",})
df.columns = df.columns.str.lower()

# keep missing bus line
df["line"] = df["line"].fillna("Unknown") 

# time format
df["date"]  = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")
df["quarter"]  = df["date"].dt.year.astype(str) + "-Q" + df["date"].dt.quarter.astype(str)
df["hour"]  = df["time"].str[:2].astype(int)

# new identifiers
df["is_delay"] = (df["min_delay"] > 0).astype(int)
df["source_id"] = MODE + "-" + df["source_id"].astype(str)


df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45475 entries, 0 to 45474
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   source_id  45475 non-null  object        
 1   date       45475 non-null  datetime64[ns]
 2   time       45475 non-null  object        
 3   day        45475 non-null  object        
 4   station    45475 non-null  object        
 5   code       45475 non-null  object        
 6   min_delay  45475 non-null  int64         
 7   min_gap    45475 non-null  int64         
 8   bound      45475 non-null  object        
 9   line       45475 non-null  object        
 10  vehicle    45475 non-null  int64         
 11  month      45475 non-null  object        
 12  quarter    45475 non-null  object        
 13  hour       45475 non-null  int64         
 14  is_delay   45475 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(9)
memory usage: 5.2+ MB


In [10]:
# load lookup table and do basic cleaning
lookup = pd.read_csv(LOOKUP, encoding="utf-8")
lookup.columns = lookup.columns.str.lower()
lookup = lookup[["code", "description"]].drop_duplicates("code")

# join table on "code"
df = df.merge(lookup, on="code", how="left")
df["description"] = df["description"].fillna("Unknown")

df.head()

,source_id,date,time,day,station,code,min_delay,min_gap,bound,line,vehicle,month,quarter,hour,is_delay,description
0,subway-1,2025-01-01,02:10,Wednesday,BATHURST STATION,MUSAN,5,9,E,BD,5227,2025-01,2025-Q1,2,1,UNSANITARY VEHICLE
1,subway-2,2025-01-01,02:30,Wednesday,DUNDAS STATION,MUIRS,0,0,Unknown,YU,0,2025-01,2025-Q1,2,0,INJURED/ILL CUSTOMER IN STATION â MEDICAL AI...
2,subway-3,2025-01-01,02:32,Wednesday,BROADVIEW STATION,PUMST,0,0,E,BD,0,2025-01,2025-Q1,2,0,ANY STAIRWAY INCIDENT â TRANSPORTED OR MEDIC...
3,subway-4,2025-01-01,02:58,Wednesday,KEELE STATION,EUSC,0,0,W,BD,5293,2025-01,2025-Q1,2,0,SPEED CONTROL EQUIPMENT
4,subway-5,2025-01-01,02:58,Wednesday,COXWELL STATION,SUAE,0,0,Unknown,BD,0,2025-01,2025-Q1,2,0,ASSAULT / EMPLOYEE INVOLVED


In [11]:
df.tail()

,source_id,date,time,day,station,code,min_delay,min_gap,bound,line,vehicle,month,quarter,hour,is_delay,description
45470,subway-45471,2026-08-31,00:26,Monday,ST GEORGE BD STATION,MUIR,10,17,W,BD,5019,2026-08,2026-Q3,0,1,INJURED/ILL CUSTOMER ON TRAIN â MEDICAL AID ...
45471,subway-45472,2026-08-31,00:36,Monday,DUNDAS WEST STATION,MUIS,0,0,Unknown,BD,0,2026-08,2026-Q3,0,0,INJURED/ILL CUSTOMER IN STATION - TRANSPORTED
45472,subway-45473,2026-08-31,06:10,Monday,DON MILLS STATION,PUOPO,4,9,W,SHP,6176,2026-08,2026-Q3,6,1,OPTO (COMMUNICATIONS) TRAIN DOOR MONITORING
45473,subway-45474,2026-08-31,08:12,Monday,DON MILLS STATION,MUIS,0,0,Unknown,SHP,0,2026-08,2026-Q3,8,0,INJURED/ILL CUSTOMER IN STATION - TRANSPORTED
45474,subway-45475,2026-08-31,16:27,Monday,SHEPPARD-YONGE STATION,PUSI,0,0,W,SHP,6161,2026-08,2026-Q3,16,0,SIGNAL OR RELATED COMPONENTS FAILURE


In [12]:
# sqlite
conn = sqlite3.connect(DB)
df.to_sql(f"{MODE}_delays", conn, if_exists="replace", index=False)

groups = {"month": "month", "hour": "hour", "reason": "description"}
for name, col in groups.items():
    conn.execute(f"DROP VIEW IF EXISTS view_{MODE}_by_{name}")
    conn.execute(f"""
    CREATE VIEW view_{MODE}_by_{name} AS
    SELECT {col},
           COUNT(*) AS incidents,
           SUM(min_delay) AS total_delay_min,
           ROUND(AVG(CASE WHEN is_delay = 1 THEN min_delay END), 2) AS avg_delay_min
    FROM {MODE}_delays
    GROUP BY {col}""")
conn.commit()

print(f"{MODE} view created successfully.")
conn.close()

subway view created successfully.
